# ILPCSR Chunk Deduplication — Analysis & Action

Investigates the 605 redundant chunk embeddings found in `neondb_explore.ipynb`. Duplicate definition: same `paragraph_id` **AND** character-for-character identical `chunk_text`. Caused by the notebook ingestion path running without intra-paragraph dedup.

**Sections**
1. Setup
2. Analysis — inspect duplicate groups with full paragraph context
3. Action — SELECT → SAVE rollback → DELETE → REINDEX

## 1 · Setup

In [ ]:
import os
import psycopg2
import psycopg2.extras
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

conn = psycopg2.connect(
    dbname=os.environ['DB_NAME'],
    user=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'],
    host=os.environ['DB_HOST'],
    port=os.environ['DB_PORT'],
    sslmode='require'
)

print('Connected:', conn.status == psycopg2.extensions.STATUS_READY)

## 2 · Analysis — duplicate groups with full paragraph context

For each group of duplicated chunks, shows: case (`doc_id`), paragraph label, full paragraph text, the duplicated chunk text, and all chunk_ids sharing that text. Use this section to confirm these are genuine duplicates before running any deletes.

In [ ]:
# All duplicate groups with paragraph context
# Groups by (paragraph_id, chunk_text); fetches full paragraph for readability.

cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("""
    WITH dup_groups AS (
        SELECT
            cc.paragraph_id,
            cc.chunk_text,
            array_agg(cc.chunk_id::text ORDER BY cc.chunk_id::text) AS chunk_ids,
            COUNT(*) AS copy_count
        FROM ilpcsr_content_chunks cc
        GROUP BY cc.paragraph_id, cc.chunk_text
        HAVING COUNT(*) > 1
    )
    SELECT
        p.doc_id,
        p.metadata_label,
        dg.paragraph_id,
        dg.copy_count,
        dg.chunk_ids,
        dg.chunk_text,
        p.full_text AS paragraph_full_text
    FROM dup_groups dg
    JOIN ilpcsr_paragraphs p ON p.paragraph_id = dg.paragraph_id
    ORDER BY p.doc_id, dg.paragraph_id
""")
dup_groups = cur.fetchall()
cur.close()

print(f"Duplicate groups : {len(dup_groups)}")
print(f"Cases affected   : {len(set(r['doc_id'] for r in dup_groups))}")
print(f"Redundant rows   : {sum(r['copy_count'] - 1 for r in dup_groups)}")

In [ ]:
# Distribution by paragraph label
label_counts = {}
for r in dup_groups:
    label = r['metadata_label'] or 'None'
    label_counts[label] = label_counts.get(label, 0) + (r['copy_count'] - 1)

print("Redundant chunks by paragraph label:")
for label, cnt in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f"  {label:<40} {cnt}")

In [ ]:
# Full paragraph text as stored in ilpcsr_paragraphs, for each duplicate group.
# No truncation, no chunk highlights — just the raw paragraph.
# Change SAMPLE_N to read through more groups.

SAMPLE_N = 5

for i, r in enumerate(dup_groups[:SAMPLE_N]):
    print('=' * 72)
    print(f"Group {i+1} of {len(dup_groups)}")
    print(f"  doc_id       : {r['doc_id']}")
    print(f"  paragraph_id : {r['paragraph_id']}")
    print(f"  label        : {r['metadata_label']}")
    print(f"  copies       : {r['copy_count']}")
    print()
    print(r['paragraph_full_text'])
    print()

print('=' * 72)
print(f"Showing {min(SAMPLE_N, len(dup_groups))} of {len(dup_groups)} groups.")

## 3 · Action — SELECT → SAVE → DELETE → REINDEX

Run only after reviewing the analysis above. Cells must be run **in order**: 3.1 populates `rows_to_delete`; 3.2 writes the rollback SQL from that variable; 3.3 deletes using that same variable (no re-query); 3.4 rebuilds the HNSW index.

In [ ]:
# 3.1 · SELECT — fetch full rows to delete (chunk + embedding)
# Stores in rows_to_delete; 3.2 and 3.3 use this variable — do NOT re-run this after 3.3.

_cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
_cur.execute("""
    WITH keepers AS (
        SELECT paragraph_id, chunk_text, MIN(chunk_id::text) AS keep_chunk_id
        FROM ilpcsr_content_chunks
        GROUP BY paragraph_id, chunk_text
        HAVING COUNT(*) > 1
    )
    SELECT
        cc.chunk_id,
        cc.paragraph_id,
        cc.chunk_text,
        ce.chunk_emb_id,
        ce.emb_model_id,
        ce.embedding
    FROM ilpcsr_content_chunks cc
    JOIN ilpcsr_chunk_embeddings ce ON ce.chunk_id = cc.chunk_id
    JOIN keepers k
        ON  k.paragraph_id = cc.paragraph_id
        AND k.chunk_text   = cc.chunk_text
    WHERE cc.chunk_id::text <> k.keep_chunk_id
""")
rows_to_delete = _cur.fetchall()
_cur.close()

print(f"Rows to delete: {len(rows_to_delete)}")
if rows_to_delete:
    r = rows_to_delete[0]
    emb = r['embedding']
    emb_len = len(emb) if isinstance(emb, (list, tuple)) else '?'
    emb_preview = emb[:3] if isinstance(emb, (list, tuple)) else str(emb)[:50]
    print(f"\nSample row:")
    print(f"  chunk_id    : {r['chunk_id']}")
    print(f"  paragraph_id: {r['paragraph_id']}")
    print(f"  chunk_emb_id: {r['chunk_emb_id']}")
    print(f"  emb_model_id: {r['emb_model_id']}")
    print(f"  chunk_text  : {repr(str(r['chunk_text'])[:80])}...")
    print(f"  embedding   : {emb_preview}... ({emb_len} dims)")

In [ ]:
# 3.2 · SAVE — write full rollback SQL from rows_to_delete (run BEFORE 3.3)
# Captures every column of every row from both tables for exact restoration.

from pathlib import Path

def _sql_str(s):
    return "'" + str(s).replace("'", "''") + "'"

def _fmt_emb(emb):
    if isinstance(emb, (list, tuple)):
        return "'[" + ','.join(str(x) for x in emb) + "]'::vector"
    return "'" + str(emb).replace("'", "''") + "'::vector"

_cwd = Path.cwd()
_ws = (_cwd / '.agent_workspace') if (_cwd / '.agent_workspace').exists() else (_cwd.parent / '.agent_workspace')
rollback_path = str(_ws / 'rollback_2.sql')

lines = [
    f"-- Rollback: restore {len(rows_to_delete)} deleted duplicate ilpcsr chunks",
    "-- Generated: 2026-06-12",
    "-- To restore: run against NeonDB via psql or pgAdmin",
    "-- Tables: ilpcsr_content_chunks, ilpcsr_chunk_embeddings",
    "", "BEGIN;", "",
    "-- 1. Restore content_chunks rows (parent table first)",
    "INSERT INTO ilpcsr_content_chunks (chunk_id, paragraph_id, chunk_text) VALUES",
]
lines.append(',\n'.join(
    f"  ({_sql_str(r['chunk_id'])}, {_sql_str(r['paragraph_id'])}, {_sql_str(r['chunk_text'])})"
    for r in rows_to_delete
) + ';')
lines += [
    "",
    "-- 2. Restore chunk_embeddings rows (child table)",
    "INSERT INTO ilpcsr_chunk_embeddings (chunk_emb_id, chunk_id, emb_model_id, embedding) VALUES",
]
lines.append(',\n'.join(
    f"  ({_sql_str(r['chunk_emb_id'])}, {_sql_str(r['chunk_id'])}, {int(r['emb_model_id'])}, {_fmt_emb(r['embedding'])})"
    for r in rows_to_delete
) + ';')
lines += ["", "COMMIT;"]

with open(rollback_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

size_kb = os.path.getsize(rollback_path) / 1024
print(f"Rollback SQL written : {rollback_path}")
print(f"Size                 : {size_kb:.1f} KB")
print(f"Rows captured        : {len(rows_to_delete)}")

In [ ]:
# 3.3 · DELETE — transactional; uses exactly rows_to_delete from 3.1 (no re-query)
# Deletes from ilpcsr_content_chunks; ON DELETE CASCADE removes ilpcsr_chunk_embeddings automatically.

_chunk_ids = [str(r['chunk_id']) for r in rows_to_delete]

try:
    _cur = conn.cursor()
    _cur.execute(
        "DELETE FROM ilpcsr_content_chunks WHERE chunk_id::text = ANY(%s)",
        (_chunk_ids,)
    )
    _deleted = _cur.rowcount
    _cur.close()
    conn.commit()
    print(f"Deleted {_deleted} rows from ilpcsr_content_chunks")
    print("CASCADE removed corresponding rows from ilpcsr_chunk_embeddings")
except Exception as e:
    conn.rollback()
    print(f"ERROR — rolled back. DB is unchanged.\n{e}")
    raise

In [ ]:
# 3.4 · REINDEX — rebuild HNSW index after deletion (non-locking, concurrent)
# REINDEX CONCURRENTLY cannot run inside a transaction block — uses separate autocommit connection.

_rc = psycopg2.connect(
    dbname=os.environ['DB_NAME'], user=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'], host=os.environ['DB_HOST'],
    port=os.environ['DB_PORT'], sslmode='require'
)
_rc.autocommit = True
try:
    _cur = _rc.cursor()
    print("Running REINDEX INDEX CONCURRENTLY idx_chunks_emb_model_1 ...")
    _cur.execute("REINDEX INDEX CONCURRENTLY idx_chunks_emb_model_1")
    _cur.close()
    print("REINDEX complete.")
finally:
    _rc.close()

## 4 · Verification — chunks without embeddings are exact duplicates

Confirms that every chunk currently in `ilpcsr_content_chunks` with no row in `ilpcsr_chunk_embeddings` is a char-for-char duplicate of another chunk sharing the same `paragraph_id`. Expected result after dedup: zero unembedded chunks, or all unembedded chunks accounted for by duplication.

In [ ]:
# 4 · Find chunks with no embedding row, then verify each is a char-for-char
# duplicate of another chunk in the same paragraph.

_cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Step 1: chunks in ilpcsr_content_chunks with no ilpcsr_chunk_embeddings row
_cur.execute("""
    SELECT cc.chunk_id, cc.paragraph_id, cc.chunk_text
    FROM ilpcsr_content_chunks cc
    LEFT JOIN ilpcsr_chunk_embeddings ce ON ce.chunk_id = cc.chunk_id
    WHERE ce.chunk_emb_id IS NULL
""")
unembedded = _cur.fetchall()

print(f"Chunks with no embedding row : {len(unembedded)}")

if not unembedded:
    print("All content chunks have embeddings — dedup clean. ✓")
else:
    unembedded_ids = [str(r['chunk_id']) for r in unembedded]

    # Step 2: for each unembedded chunk, check if an exact duplicate exists
    # in the same paragraph (different chunk_id, identical chunk_text)
    _cur.execute("""
        SELECT
            a.chunk_id   AS unembedded_chunk_id,
            a.paragraph_id,
            a.chunk_text,
            b.chunk_id   AS duplicate_of
        FROM ilpcsr_content_chunks a
        JOIN ilpcsr_content_chunks b
          ON  b.paragraph_id = a.paragraph_id
          AND b.chunk_text   = a.chunk_text
          AND b.chunk_id    <> a.chunk_id
        WHERE a.chunk_id::text = ANY(%s)
    """, (unembedded_ids,))
    confirmed_dups = _cur.fetchall()
    confirmed_ids  = {str(r['unembedded_chunk_id']) for r in confirmed_dups}

    not_dup_ids = set(unembedded_ids) - confirmed_ids

    print(f"  Confirmed exact duplicates in same para : {len(confirmed_ids)}")
    print(f"  NOT a duplicate (needs investigation)   : {len(not_dup_ids)}")

    if not_dup_ids:
        print("\n⚠️  These chunks have no embedding AND no duplicate — unexpected:")
        _cur.execute("""
            SELECT chunk_id, paragraph_id, chunk_text
            FROM ilpcsr_content_chunks
            WHERE chunk_id::text = ANY(%s)
        """, (list(not_dup_ids),))
        for r in _cur.fetchall():
            print(f"\n  chunk_id     : {r['chunk_id']}")
            print(f"  paragraph_id : {r['paragraph_id']}")
            print(f"  chunk_text   : {repr(str(r['chunk_text'])[:200])}")
    else:
        print("\nAll unembedded chunks are exact duplicates of a sibling chunk. ✓")

_cur.close()

In [ ]:
# 4b · For chunks that had no same-paragraph duplicate, check if an exact
# char-for-char match exists anywhere else in the same case.
# Uses ilpcsr_paragraphs to resolve chunk → doc_id.

if not not_dup_ids:
    print("No chunks to check — all unembedded chunks were same-paragraph duplicates.")
else:
    _cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    _cur.execute("""
        SELECT
            a.chunk_id        AS unembedded_chunk_id,
            pa.doc_id         AS case_id,
            a.chunk_text,
            b.chunk_id        AS duplicate_of,
            pb.paragraph_id   AS dup_paragraph_id
        FROM ilpcsr_content_chunks a
        JOIN ilpcsr_paragraphs pa ON pa.paragraph_id = a.paragraph_id
        JOIN ilpcsr_content_chunks b ON b.chunk_text = a.chunk_text
                                     AND b.chunk_id <> a.chunk_id
        JOIN ilpcsr_paragraphs pb ON pb.paragraph_id = b.paragraph_id
                                  AND pb.doc_id = pa.doc_id
        WHERE a.chunk_id::text = ANY(%s)
    """, (list(not_dup_ids),))
    case_dups = _cur.fetchall()
    _cur.close()

    case_dup_ids = {str(r['unembedded_chunk_id']) for r in case_dups}
    truly_unique = not_dup_ids - case_dup_ids

    print(f"Of the {len(not_dup_ids)} chunks with no same-paragraph duplicate:")
    print(f"  Have a duplicate elsewhere in same case : {len(case_dup_ids)}")
    print(f"  Truly unique (no duplicate anywhere)    : {len(truly_unique)}")

    if case_dups:
        print("\nSample case-level duplicate matches (up to 5):")
        for r in case_dups[:5]:
            print(f"  chunk_id={r['unembedded_chunk_id']}  case={r['case_id']}")
            print(f"    dup_of={r['duplicate_of']}  in_para={r['dup_paragraph_id']}")
            print(f"    text  ={repr(str(r['chunk_text'])[:120])}")
            print()

    if truly_unique:
        print(f"\n⚠️  {len(truly_unique)} chunks are unembedded with NO duplicate anywhere in their case:")
        for cid in sorted(truly_unique)[:20]:
            print(f"  {cid}")

In [ ]:
# 4c · Investigate the truly_unique unembedded chunks.
# Hypothesis: dedup kept MIN(chunk_id) as the keeper regardless of whether
# it had an embedding, then deleted the duplicate that *did* have one.
# This would leave a keeper with no embedding and no surviving duplicate.
#
# For each chunk in truly_unique, check:
#   1. Does its case have a row in ilpcsr_cases_embeddings? (case-level emb done)
#   2. Does its case have ANY chunk embeddings at all? (partial vs complete miss)
#   3. Does its case have OTHER chunks that ARE embedded? (confirms partial miss)

if not truly_unique:
    print("No truly_unique chunks to investigate.")
else:
    _cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    _cur.execute("""
        SELECT
            cc.chunk_id,
            cc.chunk_text,
            p.paragraph_id,
            p.doc_id,
            p.metadata_label,
            -- case-level embedding present?
            (SELECT COUNT(*) FROM ilpcsr_cases_embeddings ce
             WHERE ce.doc_id = p.doc_id) > 0          AS has_case_emb,
            -- total chunk embeddings for this case
            (SELECT COUNT(*) FROM ilpcsr_chunk_embeddings che
             JOIN ilpcsr_content_chunks cc2 ON cc2.chunk_id = che.chunk_id
             JOIN ilpcsr_paragraphs p2 ON p2.paragraph_id = cc2.paragraph_id
             WHERE p2.doc_id = p.doc_id)               AS case_chunk_emb_count,
            -- total chunks for this case
            (SELECT COUNT(*) FROM ilpcsr_content_chunks cc3
             JOIN ilpcsr_paragraphs p3 ON p3.paragraph_id = cc3.paragraph_id
             WHERE p3.doc_id = p.doc_id)               AS case_total_chunks
        FROM ilpcsr_content_chunks cc
        JOIN ilpcsr_paragraphs p ON p.paragraph_id = cc.paragraph_id
        WHERE cc.chunk_id::text = ANY(%s)
    """, (list(truly_unique),))
    inv = _cur.fetchall()
    _cur.close()

    inv_df = pd.DataFrame(inv)
    inv_df['missing_embs'] = inv_df['case_total_chunks'] - inv_df['case_chunk_emb_count']
    inv_df['pct_embedded'] = (inv_df['case_chunk_emb_count'] / inv_df['case_total_chunks'] * 100).round(1)

    # Breakdown
    cases_affected = inv_df['doc_id'].nunique()
    has_case_emb   = inv_df.groupby('doc_id')['has_case_emb'].first()
    partial_cases  = inv_df[inv_df['case_chunk_emb_count'] > 0]['doc_id'].nunique()
    zero_emb_cases = inv_df[inv_df['case_chunk_emb_count'] == 0]['doc_id'].nunique()

    print(f"Truly unique unembedded chunks : {len(truly_unique)}")
    print(f"Cases affected                 : {cases_affected}")
    print(f"  Cases WITH some chunk embeds : {partial_cases}  ← partial miss (dedup-keeper hypothesis)")
    print(f"  Cases with ZERO chunk embeds : {zero_emb_cases} ← complete miss")
    print(f"  Cases with case-level emb    : {has_case_emb.sum()}")
    print(f"  Cases WITHOUT case-level emb : {(~has_case_emb).sum()}")
    print()

    print("Per-case breakdown (chunks missing embed / total chunks / % embedded):")
    per_case = inv_df.drop_duplicates('doc_id')[
        ['doc_id', 'has_case_emb', 'case_chunk_emb_count', 'case_total_chunks', 'missing_embs', 'pct_embedded']
    ].sort_values('pct_embedded')
    print(per_case.to_string(index=False))
    print()

    # Sample the chunks themselves
    print("Sample truly_unique chunk texts (up to 15):")
    for _, r in inv_df.head(15).iterrows():
        print(f"  [{r['doc_id']}] {r['metadata_label']:<30} {repr(str(r['chunk_text'])[:100])}")

In [ ]:
# 4d · Two pre-embedding verifications for ALL 177 unembedded chunks with no
#      same-paragraph duplicate (truly_unique 141 + case_dup_ids 36).
#
# CHECK 1: each of the 177 is the only chunk with its chunk_text in its paragraph.
# CHECK 2: for every chunk deleted by dedup (rollback_1.sql), a chunk with the
#   exact same (paragraph_id, chunk_text) still exists in DB (the keeper).

import re
from pathlib import Path

conn.rollback()

all_177 = not_dup_ids  # truly_unique (141) + case_dup_ids (36)
print(f"Total chunks to verify : {len(all_177)}")
print()

ROLLBACK_PATHS = [
    Path(r'E:\ARooSH\CS\ML\MajorProj\CaseRag\ingestion\rollback_1.sql'),
    Path(r'E:\ARooSH\CS\ML\MajorProj\CaseRag\.agent_workspace\rollback_1.sql'),
    Path(r'E:\ARooSH\CS\ML\MajorProj\CaseRag\.agent_workspace\rollback_2.sql'),
]
LINE_RE = re.compile(
    r"\s*\('([0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12})',\s*"
    r"'([0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12})',\s*"
    r"'((?:[^']|'')*)'\)",
    re.IGNORECASE
)

rollback_entries = []
rollback_path = next((p for p in ROLLBACK_PATHS if p.exists()), None)
if rollback_path:
    print(f"Parsing {rollback_path} ...")
    with open(rollback_path, 'r', encoding='utf-8') as f:
        sql_text = f.read()
    chunks_section = sql_text.split('-- 1. Restore content_chunks')[1].split('-- 2. Restore chunk_embeddings')[0]
    for line in chunks_section.splitlines():
        m = LINE_RE.match(line)
        if m:
            rollback_entries.append((m.group(1), m.group(2), m.group(3).replace("''", "'")))
    print(f"Parsed {len(rollback_entries)} deleted chunk entries from rollback")
else:
    print("Rollback SQL not found.")

print()
_cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# ── CHECK 1 ──────────────────────────────────────────────────────────────────
print("=== CHECK 1: all 177 chunks are the sole copy in their own paragraph ===")
_cur.execute("""
    SELECT COUNT(*) AS violators
    FROM (
        SELECT cc.paragraph_id, cc.chunk_text
        FROM ilpcsr_content_chunks cc
        WHERE cc.chunk_id::text = ANY(%s)
    ) tq
    JOIN ilpcsr_content_chunks cc2
      ON  cc2.paragraph_id = tq.paragraph_id
      AND cc2.chunk_text   = tq.chunk_text
    GROUP BY tq.paragraph_id, tq.chunk_text
    HAVING COUNT(*) > 1
""", (list(all_177),))
violations = _cur.fetchall()
if not violations:
    print(f"  PASS — all {len(all_177)} chunks are the sole copy in their paragraph. ✓")
else:
    print(f"  FAIL — {len(violations)} (paragraph, chunk_text) groups have more than one copy. ✗")

print()

# ── CHECK 2 ──────────────────────────────────────────────────────────────────
print("=== CHECK 2: every rollback-deleted chunk has its keeper in DB ===")
if rollback_entries:
    rollback_para_ids = list({para_id for _, para_id, _ in rollback_entries})
    _cur.execute("""
        SELECT cc.paragraph_id::text AS para_id, cc.chunk_text
        FROM ilpcsr_content_chunks cc
        WHERE cc.paragraph_id::text = ANY(%s)
    """, (rollback_para_ids,))
    in_db = {(r['para_id'], r['chunk_text']) for r in _cur.fetchall()}

    missing_keepers = [
        (chunk_id, para_id, chunk_text)
        for chunk_id, para_id, chunk_text in rollback_entries
        if (para_id, chunk_text) not in in_db
    ]

    print(f"  Rollback entries checked : {len(rollback_entries)}")
    print(f"  Keepers found in DB      : {len(rollback_entries) - len(missing_keepers)}")
    print(f"  Missing keepers ⚠️        : {len(missing_keepers)}")

    if not missing_keepers:
        print("  PASS — every deleted chunk has a surviving keeper in the same paragraph. ✓")
    else:
        print("  FAIL — these (paragraph, chunk_text) pairs have no surviving chunk: ✗")
        for chunk_id, para_id, ct in missing_keepers[:10]:
            print(f"    deleted_chunk={chunk_id}  para={para_id}")
            print(f"    text={repr(ct[:100])}")
else:
    print("  No rollback entries to check.")

_cur.close()
print()
print("Both PASS → safe to embed all 177 chunks.")